# Cost training: Gamma regression for LLM inference costs

**Location:** Save this notebook in `2.1_Cost_Training/` at the repository root, alongside `2_Router_Training/`.

**Purpose:** Compare L2-regularized Gamma regression against Ridge regression and a training-mean baseline for predicting RouterBench inference costs for 11 LLMs. Evaluate both character TF-IDF and E5-plus-four-features using the same prepared splits as the team.

This notebook **predicts costs only**; it does not train the five-class score classifiers, perform routing, or evaluate on the test split.

For each model $m$, fit a positive conditional mean:

$$
\mathbb{E}[c_{im}\mid x_i] = \bar c_m\exp\!\left(\alpha_m + \boldsymbol{\beta}_m^T\phi(x_i)\right).
$$

Targets are normalized by their training mean, then predictions are converted back to dollar costs. `GammaRegressor` minimizes Gamma deviance with L2 coefficient regularization. The Gamma hyperparameters in this notebook are illustrative and **not cross-validated**.

**Prerequisite:** Complete the TF-IDF and E5 preparation notebooks, producing `1_data_preparation/data/processed/`.


## 1. Configuration and dependencies

Install dependencies in the same Python environment used for data preparation:

```bash
python -m pip install numpy pandas scipy scikit-learn joblib ipykernel
```

Store this notebook in `2.1_Cost_Training/` and run from anywhere **inside the project tree**. The code locates your processed input files regardless of whether VS Code starts Jupyter at the repository root or in the notebook folder. All experiment outputs are stored under `2.1_Cost_Training/experiment_outputs/gamma_cost_regression/`. The code locates either supported processed-data layout. Start with `RUN_MODE="smoke"` to verify compatibility, then set `"full"` and restart/run all cells. The smoke run is **not** a scientifically meaningful comparison.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.linear_model import GammaRegressor, Ridge
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error

RUN_MODE = "smoke"  # change to "full" for final comparative experiment
SEED = 42
DATA_PREPARATION_DIR = None  # optional path to folder containing data/
OUTPUT_BASE_OVERRIDE = None
SMOKE_LIMITS = {"train": 512, "calibration": 128, "validation": 256}

# Gamma has a different alpha scale from Ridge: these are independently chosen
# starting configurations, NOT a tuned comparison.
GAMMA_ALPHA = {"tfidf": 1.0, "e5_388": 1.0}
RIDGE_ALPHA = {"tfidf": 10.0, "e5_388": 1.0}  # teammate's cost baseline
GAMMA_MAX_ITER = 600

assert RUN_MODE in ("smoke", "full")
print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.9.1


## 2. Locate and load exactly the same prepared datasets

The TF-IDF feature matrix saved by Step 1 contains a final length column. Like your teammate's modeling notebook, we **remove** that length column for the pure-TF-IDF comparison. E5 uses all 388 dimensions.

In [2]:
def resolve_preparation_dir(override=None):
    requested = override or os.environ.get("ERDOS_DATA_DIR")
    if requested is not None:
        candidates = [Path(requested).expanduser().resolve()]
    else:
        working_dir = Path.cwd().resolve()
        candidates = [candidate for parent in (working_dir, *working_dir.parents)
                      for candidate in (parent, parent / "1_data_preparation")]
    required = (
        Path("data/processed/char_tfidf_50k/preprocessing.joblib"),
        Path("data/processed/multilingual_e5_small_chunk480/embedding_metadata.json"),
    )
    for candidate in dict.fromkeys(candidates):
        if all((candidate / rel).is_file() for rel in required):
            return candidate
    raise FileNotFoundError("Could not find processed files. Set DATA_PREPARATION_DIR to folder containing data/.")

preparation_dir = resolve_preparation_dir(DATA_PREPARATION_DIR)
table_dir = preparation_dir / "data/processed/char_tfidf_50k"
e5_dir = preparation_dir / "data/processed/multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (Path(OUTPUT_BASE_OVERRIDE).expanduser().resolve() if OUTPUT_BASE_OVERRIDE
               else project_dir / "2.1_Cost_Training/experiment_outputs/gamma_cost_regression")
run_dir = output_base / f"{RUN_MODE}_seed{SEED}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}"
run_dir.mkdir(parents=True, exist_ok=False)
print("Prepared data:", preparation_dir)
print("Outputs:", run_dir)

Prepared data: /Users/haotanhan/Documents/Career/DataScience/ErdosBootCamp/Project/fall-2026-right-for-less/fall-2026-right-for-less/1_data_preparation
Outputs: /Users/haotanhan/Documents/Career/DataScience/ErdosBootCamp/Project/fall-2026-right-for-less/fall-2026-right-for-less/2.1_Cost_Training/experiment_outputs/gamma_cost_regression/smoke_seed42_20261016T040016_362582Z


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
model_names = list(preprocessing["model_names"])
n_tfidf = len(preprocessing["tfidf_vectorizer"].vocabulary_)
SPLITS = ("train", "calibration", "validation")
feature_names = {"tfidf": "Pure character TF-IDF", "e5_388": "E5 + four features"}

tables = {}
features = {name: {} for name in feature_names}
for split_index, split in enumerate(SPLITS):
    full = {key: pd.read_pickle(table_dir / f"{split}_{key}.pkl")
            for key in ("questions", "scores", "costs")}
    q, s, c = [full[key] for key in ("questions", "scores", "costs")]
    assert q.index.is_unique and q.index.equals(s.index) and q.index.equals(c.index)
    assert s.columns.tolist() == c.columns.tolist() == model_names
    assert np.isfinite(c.to_numpy(dtype=float)).all() and c.gt(0).all().all()
    tfidf = load_npz(table_dir / f"X_{split}.npz")[:, :n_tfidf].astype(np.float32).tocsr()
    e5 = np.load(e5_dir / f"X_{split}_e5_full_features.npy", allow_pickle=False)
    assert tfidf.shape == (len(q), n_tfidf)
    assert e5.shape == (len(q), 388)
    if RUN_MODE == "smoke":
        rng = np.random.default_rng(SEED + split_index)
        ix = np.sort(rng.choice(len(q), size=min(len(q), SMOKE_LIMITS[split]), replace=False))
    else:
        ix = np.arange(len(q))
    tables[split] = {key: frame.iloc[ix].copy() for key, frame in full.items()}
    features["tfidf"][split] = tfidf[ix].copy()
    features["e5_388"][split] = e5[ix].copy()
    tables[split]["questions"][["sample_id"]].to_csv(run_dir / f"{split}_rows.csv")

# Verify that the same original questions do not occur in different splits.
all_q = pd.concat([tables[split]["questions"].assign(_split=split) for split in SPLITS])
assert all_q.groupby("question_text")["_split"].nunique().le(1).all()
assert all_q.groupby("question_group")["_split"].nunique().le(1).all()

display(pd.DataFrame({"n_questions": {s: len(tables[s]["questions"]) for s in SPLITS}}))
print("Candidate LLMs:", len(model_names))
print("TF-IDF columns:", n_tfidf, " E5 columns: 388")

,n_questions
train,512
calibration,128
validation,256


Candidate LLMs: 11
TF-IDF columns: 50000  E5 columns: 388


## 3. Fit one Gamma GLM per candidate LLM

`GammaRegressor` has one scalar target, so we fit **11 independent Gamma models** per feature representation. The Ridge comparator is the teammate's multi-output approach, trained on the same normalized costs. Both models are evaluated after restoring dollar units.

Important: the chosen Gamma penalty is a **starting value**, not cross-validated. See the optional tuning section below if your goal is a rigorously tuned comparison.

In [4]:
y_train = tables["train"]["costs"].to_numpy(dtype=np.float64)
cost_means = y_train.mean(axis=0)
assert np.all(cost_means > 0)
y_normalized = y_train / cost_means[None, :]

predictions = {}  # (method, feature, split) -> pandas DataFrame of predicted USD costs
models = {}
fit_rows = []
for feature in feature_names:
    X_train = features[feature]["train"]
    gamma_models = []
    for j, model_name in enumerate(model_names):
        glm = GammaRegressor(alpha=GAMMA_ALPHA[feature], max_iter=GAMMA_MAX_ITER, tol=1e-6)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            glm.fit(X_train, y_normalized[:, j])
        converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
        if not converged:
            warnings.warn(f"Gamma solver warning for {feature}, {model_name}; inspect convergence.")
        gamma_models.append(glm)
        fit_rows.append({"Features": feature, "LLM": model_name,
                         "Gamma_alpha": GAMMA_ALPHA[feature], "Iterations": int(glm.n_iter_),
                         "Converged_no_warning": converged})
        print(f"Gamma {feature}: {j+1}/{len(model_names)}")
    ridge = Ridge(alpha=RIDGE_ALPHA[feature], solver="lsqr", tol=1e-3,
                  max_iter=1000).fit(X_train, y_normalized)
    models[feature] = {"Gamma": gamma_models, "Ridge": ridge}
    for split in ("calibration", "validation"):
        X = features[feature][split]
        gamma_usd = np.column_stack([g.predict(X) for g in gamma_models]) * cost_means[None, :]
        ridge_usd = np.maximum(ridge.predict(X), 1e-6) * cost_means[None, :]
        mean_usd = np.broadcast_to(cost_means, gamma_usd.shape).copy()
        for name, values in [("Gamma", gamma_usd), ("Ridge", ridge_usd),
                             ("Training mean", mean_usd)]:
            assert values.shape == (len(tables[split]["questions"]), len(model_names))
            assert np.isfinite(values).all() and (values > 0).all()
            predictions[(name, feature, split)] = pd.DataFrame(
                values, index=tables[split]["costs"].index, columns=model_names
            )

display(pd.DataFrame(fit_rows).groupby("Features")[["Iterations", "Converged_no_warning"]].agg({"Iterations": "max", "Converged_no_warning": "all"}))

Gamma tfidf: 1/11
Gamma tfidf: 2/11
Gamma tfidf: 3/11
Gamma tfidf: 4/11
Gamma tfidf: 5/11
Gamma tfidf: 6/11
Gamma tfidf: 7/11
Gamma tfidf: 8/11
Gamma tfidf: 9/11
Gamma tfidf: 10/11
Gamma tfidf: 11/11
Gamma e5_388: 1/11
Gamma e5_388: 2/11
Gamma e5_388: 3/11
Gamma e5_388: 4/11
Gamma e5_388: 5/11
Gamma e5_388: 6/11
Gamma e5_388: 7/11
Gamma e5_388: 8/11
Gamma e5_388: 9/11
Gamma e5_388: 10/11
Gamma e5_388: 11/11


,Iterations,Converged_no_warning
Features,,
e5_388,12,True
tfidf,7,True


## 4. Compare cost prediction on validation

As in the teammate's README, compute each metric separately for each LLM and **average the 11 values equally**. WAPE is the average of per-model ratios (not pooled WAPE). All costs remain in USD.

In [5]:
def cost_metrics(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    assert y.shape == pred.shape
    mse_each = np.mean((pred-y)**2, axis=0)
    mae_each = np.mean(np.abs(pred-y), axis=0)
    wape_each = np.sum(np.abs(pred-y), axis=0) / np.sum(np.abs(y), axis=0)
    return {"Cost MSE (USD²)": float(mse_each.mean()),
            "Cost MAE (USD)": float(mae_each.mean()),
            "Cost WAPE (%)": float(100*wape_each.mean())}

rows = []
for (method, feature, split), frame in predictions.items():
    if split != "validation":
        continue
    rows.append({"Method": method, "Features": feature,
                 **cost_metrics(tables[split]["costs"].to_numpy(), frame.to_numpy())})
metrics = pd.DataFrame(rows).sort_values(["Cost WAPE (%)", "Cost MAE (USD)"])
display(metrics.style.format({"Cost MSE (USD²)": "{:.4e}",
                              "Cost MAE (USD)": "${:.8f}", "Cost WAPE (%)": "{:.2f}"}))
metrics.to_csv(run_dir / "validation_cost_metrics.csv", index=False)

,Method,Features,Cost MSE (USD²),Cost MAE (USD),Cost WAPE (%)
4,Ridge,e5_388,8.5268e-07,$0.00017927,15.27
1,Ridge,tfidf,1.0923e-06,$0.00026962,27.18
3,Gamma,e5_388,1.2428e-06,$0.00030155,29.25
0,Gamma,tfidf,2.2479e-06,$0.00059069,58.82
2,Training mean,tfidf,2.4366e-06,$0.00064461,63.20
5,Training mean,e5_388,2.4366e-06,$0.00064461,63.20


### Per-LLM diagnostic table

Average metrics can hide poor prediction for particular models. Check individual LLMs, especially those with unusually large or small training costs.

In [6]:
detailed = []
y_valid = tables["validation"]["costs"].to_numpy(dtype=float)
for (method, feature, split), frame in predictions.items():
    if split != "validation":
        continue
    errors = np.abs(frame.to_numpy() - y_valid)
    for j, model in enumerate(model_names):
        detailed.append({"Method": method, "Features": feature, "LLM": model,
                         "MAE (USD)": errors[:, j].mean(),
                         "WAPE (%)": 100 * errors[:, j].sum() / y_valid[:, j].sum()})
per_llm = pd.DataFrame(detailed)
display(per_llm.sort_values("WAPE (%)").head(20))
per_llm.to_csv(run_dir / "validation_cost_metrics_per_llm.csv", index=False)

,Method,Features,LLM,MAE (USD),WAPE (%)
53,Ridge,e5_388,mistralai/mixtral-8x7b-chat,0.000016,11.013953
51,Ridge,e5_388,meta/llama-2-70b-chat,0.000023,11.242438
50,Ridge,e5_388,meta/code-llama-instruct-34b-chat,0.000020,11.334439
52,Ridge,e5_388,mistralai/mistral-7b-chat,0.000006,12.304206
44,Ridge,e5_388,WizardLM/WizardLM-13B-V1.2,0.000011,13.995983
54,Ridge,e5_388,zero-one-ai/Yi-34B-Chat,0.000029,15.090544
48,Ridge,e5_388,gpt-3.5-turbo-1106,0.000039,15.316375
45,Ridge,e5_388,claude-instant-v1,0.000042,16.818403
46,Ridge,e5_388,claude-v1,0.000400,17.255726
47,Ridge,e5_388,claude-v2,0.000461,17.978516


## 5. Save models and predictions for later routing experiments

The calibration split is **not used to fit these cost models**. Predictions for calibration and validation are saved, along with model names, cost normalization values, and configuration. Another teammate can load these files for the routing stage without rerunning Gamma fitting.

**Git note:** The resulting model/prediction files can be large. Add `2.1_Cost_Training/experiment_outputs/` to the repository `.gitignore` before staging notebook work if you do not intend to version these outputs.

In [7]:
for (method, feature, split), frame in predictions.items():
    key = method.lower().replace(" ", "_")
    frame.to_pickle(run_dir / f"cost_predictions_{key}_{feature}_{split}.pkl")

joblib.dump(models, run_dir / "cost_models_gamma_and_ridge.joblib")
metadata = {
    "run_mode": RUN_MODE, "seed": SEED, "model_names": model_names,
    "features": feature_names, "gamma_alpha": GAMMA_ALPHA,
    "ridge_alpha": RIDGE_ALPHA, "cost_means": cost_means.tolist(),
    "prepared_data_directory": str(preparation_dir),
    "sklearn_version": sklearn.__version__,
    "metrics": "Unweighted mean across 11 per-LLM metrics; WAPE expressed as a percentage",
    "test_evaluated": False,
    "gamma_alpha_tuned": False,
}
(run_dir / "experiment_manifest.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved results to:", run_dir)

Saved results to: /Users/haotanhan/Documents/Career/DataScience/ErdosBootCamp/Project/fall-2026-right-for-less/fall-2026-right-for-less/2.1_Cost_Training/experiment_outputs/gamma_cost_regression/smoke_seed42_20261016T040016_362582Z


## 6. Optional next step: hyperparameter selection

For a fairer comparison with your teammate's methods, choose `GammaRegressor(alpha=...)` **only using training folds grouped by `question_group`**, then evaluate once on validation. A sensible candidate grid is `alpha in [0.01, 0.1, 1, 10]`, but this can require **many costly fits** (11 LLMs × 2 representations × 4 penalties × 3 folds = 264 Gamma fits). The notebook deliberately leaves this expensive step optional. Avoid selecting alpha by repeatedly optimizing validation metrics if validation is being reserved for model or policy selection.

**Interpretation caveats:** A Gamma GLM assumes the conditional mean has a log-linear form and a Gamma-like mean–variance relationship. It is not guaranteed to beat Ridge or kNN. Success here means **lower held-out cost error**, not necessarily a better cost–accuracy routing frontier. Keep the test set untouched until your team has selected its final policy.